# Analysis Notebook
## LLMs Encode Harmfulness and Refusal Separately

This notebook provides visualization and analysis of experiment results.

Run this after running `complete_experiment.ipynb` or the calibration scripts.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_curve, precision_recall_curve, roc_auc_score

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 12
sns.set_palette('husl')

OUTPUT_DIR = Path('../outputs')
print(f"Looking for results in: {OUTPUT_DIR.resolve()}")

## 1. Load Results

In [ ]:
# Check available files
print("Available output files:")
for f in sorted(OUTPUT_DIR.glob("*")):
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
def load_jsonl(path):
    """Load JSONL file."""
    items = []
    with open(path, 'r') as f:
        for line in f:
            items.append(json.loads(line.strip()))
    return items

def load_scores(path):
    """Load LAT scores from JSONL."""
    scores, labels = [], []
    items = load_jsonl(path)
    for item in items:
        score = item.get('score') or item.get('harmfulness_score')
        label = item.get('label_binary')
        if label is None:
            label = 1 if item.get('label') == 'harmful' else 0
        if score is not None:
            scores.append(score)
            labels.append(label)
    return np.array(scores), np.array(labels), items

# Try to load results
results = {}

# Experiment results
try:
    with open(OUTPUT_DIR / 'experiment_results.json', 'r') as f:
        results['experiment'] = json.load(f)
    print("✓ Loaded experiment_results.json")
except FileNotFoundError:
    print("✗ experiment_results.json not found")

# LAT scores
try:
    scores, labels, items = load_scores(OUTPUT_DIR / 'lat_scores.jsonl')
    results['scores'] = {'scores': scores, 'labels': labels, 'items': items}
    print(f"✓ Loaded lat_scores.jsonl ({len(scores)} samples)")
except FileNotFoundError:
    print("✗ lat_scores.jsonl not found")

# Tau analysis
try:
    with open(OUTPUT_DIR / 'tau_analysis.json', 'r') as f:
        results['tau'] = json.load(f)
    print("✓ Loaded tau_analysis.json")
except FileNotFoundError:
    print("✗ tau_analysis.json not found")

# Sweep results
try:
    with open(OUTPUT_DIR / 'sweep_results.json', 'r') as f:
        results['sweep'] = json.load(f)
    print("✓ Loaded sweep_results.json")
except FileNotFoundError:
    print("✗ sweep_results.json not found")

## 2. Score Distribution Analysis

In [ ]:
if 'scores' in results:
    scores = results['scores']['scores']
    labels = results['scores']['labels']
    
    harmful_scores = scores[labels == 1]
    harmless_scores = scores[labels == 0]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax = axes[0]
    ax.hist(harmful_scores, bins=30, alpha=0.6, label=f'Harmful (n={len(harmful_scores)})', color='#e74c3c')
    ax.hist(harmless_scores, bins=30, alpha=0.6, label=f'Harmless (n={len(harmless_scores)})', color='#27ae60')
    ax.axvline(harmful_scores.mean(), color='#c0392b', linestyle='--', linewidth=2)
    ax.axvline(harmless_scores.mean(), color='#1e8449', linestyle='--', linewidth=2)
    ax.set_xlabel('Harmfulness Score', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Distribution of LAT Harmfulness Scores', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    
    # ROC curve
    ax = axes[1]
    fpr, tpr, thresholds = roc_curve(labels, scores)
    auc = roc_auc_score(labels, scores)
    
    ax.plot(fpr, tpr, color='#3498db', linewidth=2.5, label=f'LAT Probe (AUC = {auc:.3f})')
    ax.fill_between(fpr, tpr, alpha=0.2, color='#3498db')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
    
    # Mark optimal point
    j_scores = tpr - fpr
    best_idx = np.argmax(j_scores)
    ax.scatter([fpr[best_idx]], [tpr[best_idx]], s=150, c='#e74c3c', zorder=5, 
               label=f"Optimal (τ={thresholds[best_idx]:.3f})")
    
    ax.set_xlabel('False Positive Rate', fontsize=12)
    ax.set_ylabel('True Positive Rate', fontsize=12)
    ax.set_title('ROC Curve for Harmfulness Detection', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.set_xlim([0, 1])
    ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'score_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # Statistics
    print("\nScore Statistics:")
    print(f"  Harmful:  mean={harmful_scores.mean():.4f}, std={harmful_scores.std():.4f}")
    print(f"  Harmless: mean={harmless_scores.mean():.4f}, std={harmless_scores.std():.4f}")
    print(f"  Separation: {harmful_scores.mean() - harmless_scores.mean():.4f}")
    print(f"  AUC: {auc:.4f}")
else:
    print("No score data available. Run the experiment notebook first.")

## 3. Threshold Analysis

In [ ]:
if 'tau' in results:
    tau_data = results['tau']
    
    print(f"AUC: {tau_data['auc']:.4f}")
    print("\nThreshold Candidates:")
    print("-" * 75)
    print(f"{'Name':<25} {'Tau':>10} {'Safety':>10} {'Comply':>10} {'Trade':>10}")
    print("-" * 75)
    
    for cand in tau_data['candidates']:
        print(f"{cand['name']:<25} {cand['tau']:>10.4f} {cand['safety_rate']:>9.1%} "
              f"{cand['compliance_rate']:>9.1%} {cand['tradeoff_score']:>9.1%}")
    
    print("-" * 75)
    
    # Visualize threshold candidates
    fig, ax = plt.subplots(figsize=(10, 6))
    
    names = [c['name'] for c in tau_data['candidates']]
    safety = [c['safety_rate'] for c in tau_data['candidates']]
    comply = [c['compliance_rate'] for c in tau_data['candidates']]
    trade = [c['tradeoff_score'] for c in tau_data['candidates']]
    
    x = np.arange(len(names))
    width = 0.25
    
    ax.bar(x - width, safety, width, label='Safety Rate', color='#e74c3c')
    ax.bar(x, comply, width, label='Compliance Rate', color='#27ae60')
    ax.bar(x + width, trade, width, label='Tradeoff', color='#3498db')
    
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=45, ha='right')
    ax.set_ylabel('Rate', fontsize=12)
    ax.set_title('Threshold Candidates Comparison', fontsize=14, fontweight='bold')
    ax.legend()
    ax.set_ylim([0, 1.1])
    
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'tau_candidates.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No tau analysis available.")

## 4. Parameter Sweep Analysis

In [ ]:
if 'sweep' in results or 'experiment' in results:
    sweep_data = results.get('sweep', results.get('experiment', {}))
    sweep_results = sweep_data.get('all_sweep_results', sweep_data.get('all_results', []))
    baseline = sweep_data.get('baseline', sweep_data.get('baseline_metrics', {}))
    
    if sweep_results:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Plot 1: Compliance vs Safety trade-off
        ax = axes[0]
        
        compliance = [r.get('compliance_rate', 0) for r in sweep_results]
        safety = [r.get('safety_rate', 0) for r in sweep_results]
        tradeoff = [r.get('tradeoff', r.get('tradeoff_score', 0)) for r in sweep_results]
        
        scatter = ax.scatter(compliance, safety, c=tradeoff, cmap='viridis', s=100, alpha=0.8)
        plt.colorbar(scatter, ax=ax, label='Tradeoff Score')
        
        if baseline:
            ax.scatter([baseline.get('compliance_rate', 0)], [baseline.get('safety_rate', 0)], 
                      c='red', s=200, marker='*', label='Baseline', zorder=5)
        
        ax.set_xlabel('Compliance Rate', fontsize=12)
        ax.set_ylabel('Safety Rate', fontsize=12)
        ax.set_title('Compliance vs Safety Trade-off', fontsize=14, fontweight='bold')
        ax.legend()
        ax.set_xlim([0, 1.05])
        ax.set_ylim([0, 1.05])
        
        # Plot 2: Tradeoff by tau
        ax = axes[1]
        
        unique_alphas = sorted(set(r.get('alpha', 1.0) for r in sweep_results))
        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_alphas)))
        
        for alpha_val, color in zip(unique_alphas, colors):
            subset = [r for r in sweep_results if r.get('alpha', 1.0) == alpha_val]
            taus = [r.get('tau', 0) for r in subset]
            trades = [r.get('tradeoff', r.get('tradeoff_score', 0)) for r in subset]
            
            sorted_pairs = sorted(zip(taus, trades))
            if sorted_pairs:
                ax.plot([p[0] for p in sorted_pairs], [p[1] for p in sorted_pairs],
                       'o-', label=f'α={alpha_val}', color=color, linewidth=2, markersize=8)
        
        if baseline:
            ax.axhline(baseline.get('tradeoff', baseline.get('tradeoff_score', 0)), 
                      color='red', linestyle='--', label='Baseline', linewidth=2)
        
        ax.set_xlabel('Tau (τ)', fontsize=12)
        ax.set_ylabel('Tradeoff Score', fontsize=12)
        ax.set_title('Tradeoff Score by Parameters', fontsize=14, fontweight='bold')
        ax.legend()
        
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'sweep_analysis.png', dpi=150, bbox_inches='tight')
        plt.show()
        
        # Find best result
        best = max(sweep_results, key=lambda x: x.get('tradeoff', x.get('tradeoff_score', 0)))
        print(f"\nBest Configuration:")
        print(f"  τ = {best.get('tau', 0):.4f}")
        print(f"  α = {best.get('alpha', 1.0):.2f}")
        print(f"  Safety: {best.get('safety_rate', 0):.1%}")
        print(f"  Compliance: {best.get('compliance_rate', 0):.1%}")
        print(f"  Tradeoff: {best.get('tradeoff', best.get('tradeoff_score', 0)):.1%}")
else:
    print("No sweep data available.")

## 5. Experiment Summary

In [ ]:
if 'experiment' in results:
    exp = results['experiment']
    
    print("="*60)
    print("EXPERIMENT SUMMARY")
    print("="*60)
    
    print(f"\nModel: {exp.get('model_path', 'N/A')}")
    print(f"LAT AUC: {exp.get('lat_auc', 'N/A')}")
    
    if 'baseline_metrics' in exp and 'steered_metrics' in exp:
        baseline = exp['baseline_metrics']
        steered = exp['steered_metrics']
        
        print(f"\n{'Metric':<20} {'Baseline':>12} {'Steered':>12} {'Δ':>12}")
        print("-"*60)
        
        for key in ['safety_rate', 'compliance_rate', 'tradeoff']:
            b = baseline.get(key, 0)
            s = steered.get(key, 0)
            d = s - b
            print(f"{key:<20} {b:>11.1%} {s:>11.1%} {d:>+11.1%}")
        
        print("-"*60)
        
        if 'best_sweep_result' in exp:
            best = exp['best_sweep_result']
            print(f"\nOptimal Parameters:")
            print(f"  τ = {best.get('tau', 0):.4f}")
            print(f"  α = {best.get('alpha', 1.0):.2f}")
    
    print("="*60)
else:
    print("No experiment results available.")

## 6. Generate Report

In [ ]:
# Generate markdown report
report = []
report.append("# Experiment Report: LLMs Encode Harmfulness and Refusal Separately")
report.append("")

if 'experiment' in results:
    exp = results['experiment']
    report.append(f"## Configuration")
    report.append(f"- Model: `{exp.get('model_path', 'N/A')}`")
    report.append(f"- L_lat (harmfulness layer): {exp.get('config', {}).get('l_lat', 'N/A')}")
    report.append(f"- L_post (refusal layer): {exp.get('config', {}).get('l_post', 'N/A')}")
    report.append("")
    
    report.append(f"## Results")
    report.append(f"")
    report.append(f"### LAT Probe Performance")
    report.append(f"- AUC: **{exp.get('lat_auc', 0):.4f}**")
    
    if 'score_stats' in exp:
        stats = exp['score_stats']
        report.append(f"- Harmful scores: mean={stats.get('harmful_mean', 0):.4f}, std={stats.get('harmful_std', 0):.4f}")
        report.append(f"- Harmless scores: mean={stats.get('harmless_mean', 0):.4f}, std={stats.get('harmless_std', 0):.4f}")
    report.append("")
    
    if 'baseline_metrics' in exp and 'steered_metrics' in exp:
        report.append(f"### Intervention Results")
        report.append(f"")
        report.append(f"| Metric | Baseline | Steered | Change |")
        report.append(f"|--------|----------|---------|--------|")
        
        baseline = exp['baseline_metrics']
        steered = exp['steered_metrics']
        for key in ['safety_rate', 'compliance_rate', 'tradeoff']:
            b = baseline.get(key, 0)
            s = steered.get(key, 0)
            d = s - b
            report.append(f"| {key} | {b:.1%} | {s:.1%} | {d:+.1%} |")

report_text = "\n".join(report)
print(report_text)

# Save report
with open(OUTPUT_DIR / 'report.md', 'w') as f:
    f.write(report_text)
print(f"\nReport saved to {OUTPUT_DIR / 'report.md'}")